# Brain-body GPU demo

Notebook nay chay source brain-body that, FlyGym/MuJoCo, RolloutRecorder,
analysis, biomarkers va viewer export. No khong doc video hoac JSON summary
de tao chuyen dong. Ket qua la computational locomotion simulation, khong
phai biological Parkinson validation.

## Sections
1. Discover source and repository
2. Verify GPU and source data
3. Run healthy baseline
4. Run computational condition
5. Validate artifacts and download viewer bundle

Run all cells from top to bottom. A missing runtime or source file stops the
notebook with an explicit error.

## Section 1 - Discover source and repository

The main repository must contain `scripts/run_brain_body_rollout.py`. The
separate brain source is taken from `FLY_BRAIN_ROOT`, a sibling
`phase-A-clean` checkout, or cloned from the public source repository.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

def find_repo():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'scripts' / 'run_brain_body_rollout.py').is_file():
            return candidate.resolve()
    raise RuntimeError('Main repository was not found from the notebook working directory.')

repo = find_repo()
brain_root = Path(os.environ.get('FLY_BRAIN_ROOT', repo.parent / 'phase-A-clean')).expanduser().resolve()
if not brain_root.is_dir():
    clone_target = Path('/content/phase-A-clean') if Path('/content').is_dir() else repo.parent / 'phase-A-clean'
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/tuanwannafly/drosophila-pd-flygym.git',
        str(clone_target),
    ], check=True)
    brain_root = clone_target.resolve()

required = [
    brain_root / 'brain_body_bridge.py',
    brain_root / 'code' / 'run_pytorch.py',
    brain_root / 'data' / '2025_Completeness_783.csv',
    brain_root / 'data' / '2025_Connectivity_783.parquet',
    brain_root / 'data' / 'plastic_weights.pt',
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise RuntimeError('Brain source is incomplete. Missing: ' + ', '.join(missing))
output_root = repo / 'results' / 'brain_body_notebook'
output_root.mkdir(parents=True, exist_ok=True)
print('Repository:', repo)
print('Brain source:', brain_root)
print('Output root:', output_root)

## Section 2 - Verify GPU and source data

The runner selects CUDA when available and can re-execute with the brain
source environment on Windows/Linux. This cell does not install packages or
replace missing weights.

In [ ]:
brain_python_candidates = [
    Path(os.environ['FLY_BRAIN_PYTHON']).expanduser() if os.environ.get('FLY_BRAIN_PYTHON') else None,
    brain_root / '.venv' / 'Scripts' / 'python.exe',
    brain_root / '.venv' / 'bin' / 'python',
    Path(sys.executable),
]
brain_python = next((path for path in brain_python_candidates if path and path.is_file()), Path(sys.executable))
gpu_probe = subprocess.run([
    str(brain_python), '-c',
    "import torch; print(torch.__version__); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')",
], capture_output=True, text=True, check=False)
if gpu_probe.returncode != 0:
    raise RuntimeError('PyTorch is missing in the selected brain environment: ' + gpu_probe.stderr.strip())
probe_lines = gpu_probe.stdout.strip().splitlines()
if len(probe_lines) < 2 or probe_lines[1].strip().lower() != 'true':
    raise RuntimeError('CUDA is unavailable. Enable a GPU runtime or use the phase-A CUDA environment.')
print('Brain Python:', brain_python)
print('Torch:', probe_lines[0])
print('CUDA device:', probe_lines[2] if len(probe_lines) > 2 else 'unknown')
print('Required source files:', len(required))

## Section 3 - Run healthy baseline

This is the unperturbed computational baseline. The first recorded frame is
included, so the expected viewer frame count is `steps + 1`.

In [ ]:
healthy_output = output_root / 'healthy_seed_0'
healthy_command = [
    sys.executable, str(repo / 'scripts' / 'run_brain_body_rollout.py'),
    '--brain-root', str(brain_root),
    '--condition', 'healthy',
    '--seed', '0', '--steps', '1000', '--device', 'cuda',
    '--output', str(healthy_output),
]
subprocess.run(healthy_command, cwd=str(repo), check=True)
print('Healthy output:', healthy_output)

## Section 4 - Run computational condition

The condition uses the existing action/controller Disease Layer configuration.
It is a computational perturbation for comparison, not a biological disease
claim.

In [ ]:
condition_output = output_root / 'computational_pd_like_demo_seed_0'
condition_command = [
    sys.executable, str(repo / 'scripts' / 'run_brain_body_rollout.py'),
    '--brain-root', str(brain_root),
    '--condition', 'computational_pd_like_demo',
    '--disease-config', str(repo / 'configs' / 'parkinson' / 'computational_pd_like_demo.yaml'),
    '--seed', '0', '--steps', '1000', '--device', 'cuda',
    '--compare-to', str(healthy_output),
    '--output', str(condition_output),
]
subprocess.run(condition_command, cwd=str(repo), check=True)
print('Condition output:', condition_output)

## Section 5 - Validate artifacts and download viewer bundle

The checks below validate real exported data: frame count, finite values,
strict timestamps and non-zero normalized quaternions.

In [ ]:
import json
import numpy as np

def validate_run(run_dir, expected_steps):
    run_dir = Path(run_dir)
    expected = ['rollout.json', 'rollout.npz', 'viewer_pose.json', 'metadata.json', 'manifest.json', 'brain_body_manifest.json']
    for name in expected:
        assert (run_dir / name).is_file(), f'Missing {run_dir / name}'
    pose = json.loads((run_dir / 'viewer_pose.json').read_text(encoding='utf-8'))
    assert pose['frame_count'] == expected_steps + 1
    frames = pose['frames']
    times = np.asarray([frame['time'] for frame in frames], dtype=float)
    quaternions = np.asarray([frame['orientation'] for frame in frames], dtype=float)
    assert np.isfinite(times).all() and np.all(np.diff(times) > 0)
    norms = np.linalg.norm(quaternions, axis=1)
    assert np.isfinite(quaternions).all() and np.all(norms > 0)
    assert np.allclose(norms, 1.0, atol=1e-6)
    return pose

healthy_pose = validate_run(healthy_output, 1000)
condition_pose = validate_run(condition_output, 1000)
print('Healthy frames:', healthy_pose['frame_count'])
print('Condition frames:', condition_pose['frame_count'])
print('PASS: real rollout -> viewer_pose pipeline validated')

In [ ]:
healthy_bundle = healthy_output / 'viewer_bundle.zip'
condition_bundle = condition_output / 'viewer_bundle.zip'
assert healthy_bundle.is_file() and condition_bundle.is_file()
print('Healthy bundle:', healthy_bundle)
print('Condition bundle:', condition_bundle)
try:
    from google.colab import files
    files.download(str(condition_bundle))
except ModuleNotFoundError:
    from IPython.display import FileLink, display
    display(FileLink(str(condition_bundle)))

## Expected artifacts

Each run contains `rollout.json`, `rollout.npz`, `rollout.csv`,
`viewer_pose.json`, `manifest.json`, `metadata.json`, metrics, figures,
biomarkers, a report, and `viewer_bundle.zip`. Open the downloaded bundle via
an HTTP server or GitHub Pages; do not open the HTML with `file://` when the
browser blocks JSON fetches.

The outputs remain computational artifacts. They do not establish a biological
Parkinson phenotype, clinical prediction, or treatment response.